# День 09. Упражнение 00
# Регуляризация

## 0. Импорт

In [59]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import joblib

## 1. Предобработка

1. Прочитайте файл `dayofweek.csv`, который вы использовали накануне, в датафрейм.
2. Используя `train_test_split` с параметрами `test_size=0.2`, `random_state=21`, получите `X_train`, `y_train`, `X_test`, `y_test`. Используйте дополнительный параметр `stratify`.

In [60]:
df = pd.read_csv('../data/dayofweek.csv')

In [61]:
print(df.head())
print(df.columns)

   numTrials      hour  dayofweek  uid_user_0  uid_user_1  uid_user_10  \
0  -0.788667 -2.562352          4         0.0         0.0          0.0   
1  -0.756764 -2.562352          4         0.0         0.0          0.0   
2  -0.724861 -2.562352          4         0.0         0.0          0.0   
3  -0.692958 -2.562352          4         0.0         0.0          0.0   
4  -0.661055 -2.562352          4         0.0         0.0          0.0   

   uid_user_11  uid_user_12  uid_user_13  uid_user_14  ...  labname_lab02  \
0          0.0          0.0          0.0          0.0  ...            0.0   
1          0.0          0.0          0.0          0.0  ...            0.0   
2          0.0          0.0          0.0          0.0  ...            0.0   
3          0.0          0.0          0.0          0.0  ...            0.0   
4          0.0          0.0          0.0          0.0  ...            0.0   

   labname_lab03  labname_lab03s  labname_lab05s  labname_laba04  \
0            0.0        

In [62]:
# Разделение на признаки и целевую переменную
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

## 2. Регуляризация логистической регрессии

### a. Базовая регуляризация

1. Обучите базовую модель с параметрами `random_state=21`, `fit_intercept=False`.
2. Используйте стратифицированную кросс-валидацию с 10 фолдами для оценки точности модели.

Результат кода, где вы обучаете и оцениваете базовую модель, должен быть точно таким (используйте `%%time` для отображения времени выполнения ячейки):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
точность на кросс-валидации: 0.60165
Стандартное отклонение: 0.02943
```

In [63]:
%%time
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = LogisticRegression(random_state=21, fit_intercept=False, solver='lbfgs', max_iter=1000) #  penalty=None
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

train -  0.64056   |   valid -  0.65926
train -  0.63561   |   valid -  0.62222
train -  0.64468   |   valid -  0.60000
train -  0.64056   |   valid -  0.64444
train -  0.65375   |   valid -  0.60741
train -  0.62902   |   valid -  0.60000
train -  0.66117   |   valid -  0.60000
train -  0.63726   |   valid -  0.54074
train -  0.63756   |   valid -  0.66418
train -  0.64745   |   valid -  0.61194
точность на кросс-валидации: 0.61502
Стандартное отклонение: 0.03399
CPU times: user 117 ms, sys: 3.06 ms, total: 120 ms
Wall time: 119 ms


### b. Подбор параметров регуляризации

1. В следующих ячейках попробуйте разные значения параметра penalty: `none`, `l1`, `l2`. Можно менять значения solver.

In [65]:
%%time

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import numpy as np

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    
    model = LogisticRegression(random_state=21)
    
    model.fit(X_tr, y_tr)
    
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')

print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

train -  0.64303   |   valid -  0.65926
train -  0.64551   |   valid -  0.62222
train -  0.64303   |   valid -  0.60741
train -  0.64551   |   valid -  0.64444
train -  0.65622   |   valid -  0.60741
train -  0.63397   |   valid -  0.60000
train -  0.66447   |   valid -  0.60000
train -  0.63809   |   valid -  0.54074
train -  0.64003   |   valid -  0.67910
train -  0.64827   |   valid -  0.60448
Точность на кросс-валидации: 0.61651
Стандартное отклонение: 0.03627
CPU times: user 149 ms, sys: 3.21 ms, total: 152 ms
Wall time: 150 ms


In [66]:
%%time

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    
    model = LogisticRegression(random_state=21)
    
    model.fit(X_tr, y_tr)
    
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')

print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

train -  0.64303   |   valid -  0.65926
train -  0.64551   |   valid -  0.62222
train -  0.64303   |   valid -  0.60741
train -  0.64551   |   valid -  0.64444
train -  0.65622   |   valid -  0.60741
train -  0.63397   |   valid -  0.60000
train -  0.66447   |   valid -  0.60000
train -  0.63809   |   valid -  0.54074
train -  0.64003   |   valid -  0.67910
train -  0.64827   |   valid -  0.60448
Точность на кросс-валидации: 0.61651
Стандартное отклонение: 0.03627
CPU times: user 147 ms, sys: 3.14 ms, total: 150 ms
Wall time: 149 ms


## 3. Регуляризация SVM

### a. Базовая регуляризация

1. Обучите базовую модель с параметрами `probability=True`, `kernel='linear'`, `random_state=21`.
2. Используйте стратифицированную кросс-валидацию с 10 фолдами для оценки точности модели.
3. Формат вывода результатов должен быть аналогичен тому, что был для логистической регрессии.

In [67]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = SVC(probability=True, kernel='linear', random_state=21)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

train -  0.70651   |   valid -  0.68148
train -  0.68920   |   valid -  0.64444
train -  0.69744   |   valid -  0.66667
train -  0.68920   |   valid -  0.65926
train -  0.69497   |   valid -  0.63704
train -  0.68673   |   valid -  0.68148
train -  0.69827   |   valid -  0.61481
train -  0.70486   |   valid -  0.57778
train -  0.68863   |   valid -  0.72388
train -  0.71005   |   valid -  0.64179
Точность на кросс-валидации: 0.65286
Стандартное отклонение: 0.03800


### b. Подбор параметров регуляризации

1. В следующих ячейках попробуйте разные значения параметра `C`.

In [68]:
# SVM с C=0.1
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = SVC(probability=True, kernel='linear', random_state=21, C=0.1)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

train -  0.57049   |   valid -  0.55556
train -  0.56884   |   valid -  0.59259
train -  0.57543   |   valid -  0.54074
train -  0.56142   |   valid -  0.60000
train -  0.59110   |   valid -  0.57037
train -  0.57873   |   valid -  0.53333
train -  0.59687   |   valid -  0.54074
train -  0.59439   |   valid -  0.52593
train -  0.56590   |   valid -  0.58209
train -  0.58731   |   valid -  0.53731
Точность на кросс-валидации: 0.55787
Стандартное отклонение: 0.02522


In [69]:
# SVM с C=10
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = SVC(probability=True, kernel='linear', random_state=21, C=10)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

train -  0.76175   |   valid -  0.71852
train -  0.76340   |   valid -  0.71852
train -  0.76834   |   valid -  0.74074
train -  0.78071   |   valid -  0.76296
train -  0.75927   |   valid -  0.68889
train -  0.72630   |   valid -  0.74815
train -  0.76834   |   valid -  0.67407
train -  0.76340   |   valid -  0.64444
train -  0.77183   |   valid -  0.78358
train -  0.78418   |   valid -  0.70149
Точность на кросс-валидации: 0.71814
Стандартное отклонение: 0.04026


In [70]:
# SVM с C=100
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = SVC(probability=True, kernel='linear', random_state=21, C=100)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

train -  0.79143   |   valid -  0.74074
train -  0.80214   |   valid -  0.75556
train -  0.78566   |   valid -  0.75556
train -  0.80214   |   valid -  0.74815
train -  0.78566   |   valid -  0.74815
train -  0.77824   |   valid -  0.80000
train -  0.79637   |   valid -  0.71852
train -  0.80132   |   valid -  0.71852
train -  0.78501   |   valid -  0.79851
train -  0.79242   |   valid -  0.71642
Точность на кросс-валидации: 0.75001
Стандартное отклонение: 0.02849


### Сравнение моделей SVM

| C    | Средняя точность | Ст. отклонение | Комментарий  |
|------|------------------|----------------|--------------|
| 0.1  | 0.55787          | 0.02522        | Недообучена  |
| 10   | 0.71814          | 0.04026        | Оптимальна   |
| 100  | 0.75001          | 0.02849        | Переобучена  |

**Вывод:**  
Оптимальные значения C — 10 или 100.  
При низком C (0.1) модель недообучена, точность низкая.  
При большом C (100) модель начинает переобучаться (train accuracy сильно выше valid).

## 4. Дерево решений

### a. Базовая регуляризация

1. Обучите базовую модель с единственным параметром `max_depth=10` и `random_state=21`.
2. Используйте стратифицированную кросс-валидацию с 10 фолдами для оценки точности модели.
3. Формат вывода результатов должен быть аналогичен тому, что был для логистической регрессии.


In [40]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = DecisionTreeClassifier(max_depth=10, random_state=21)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

train -  0.80874   |   valid -  0.77037
train -  0.79802   |   valid -  0.70370
train -  0.81286   |   valid -  0.72593
train -  0.80049   |   valid -  0.74815
train -  0.80956   |   valid -  0.68889
train -  0.78978   |   valid -  0.74074
train -  0.80627   |   valid -  0.60741
train -  0.82688   |   valid -  0.71111
train -  0.78995   |   valid -  0.79104
train -  0.80313   |   valid -  0.70896
Точность на кросс-валидации: 0.71963
Стандартное отклонение: 0.04791


### b. Оптимизация параметров регуляризации

1. В ячейках ниже попробуйте разные значения параметра max_depth.
2. В качестве бонуса поэкспериментируйте с другими параметрами регуляризации, чтобы найти наилучшую комбинацию.

In [41]:
# DecisionTreeClassifier с max_depth=5
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = DecisionTreeClassifier(max_depth=5, random_state=21)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

train -  0.58285   |   valid -  0.60741
train -  0.57626   |   valid -  0.52593
train -  0.61253   |   valid -  0.60000
train -  0.58862   |   valid -  0.58519
train -  0.58615   |   valid -  0.51111
train -  0.56224   |   valid -  0.53333
train -  0.58120   |   valid -  0.51852
train -  0.62407   |   valid -  0.51111
train -  0.57414   |   valid -  0.56716
train -  0.56672   |   valid -  0.48507
Точность на кросс-валидации: 0.54448
Стандартное отклонение: 0.04014


In [42]:
# DecisionTreeClassifier с max_depth=7
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = DecisionTreeClassifier(max_depth=7, random_state=21)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

train -  0.69580   |   valid -  0.70370
train -  0.66529   |   valid -  0.61481
train -  0.70074   |   valid -  0.66667
train -  0.69580   |   valid -  0.68889
train -  0.70157   |   valid -  0.63704
train -  0.67436   |   valid -  0.66667
train -  0.68013   |   valid -  0.62222
train -  0.71970   |   valid -  0.59259
train -  0.66474   |   valid -  0.72388
train -  0.68287   |   valid -  0.61194
Точность на кросс-валидации: 0.65284
Стандартное отклонение: 0.04153


In [43]:
# DecisionTreeClassifier с max_depth=10
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = DecisionTreeClassifier(max_depth=10, random_state=21)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

train -  0.80874   |   valid -  0.77037
train -  0.79802   |   valid -  0.70370
train -  0.81286   |   valid -  0.72593
train -  0.80049   |   valid -  0.74815
train -  0.80956   |   valid -  0.68889
train -  0.78978   |   valid -  0.74074
train -  0.80627   |   valid -  0.60741
train -  0.82688   |   valid -  0.71111
train -  0.78995   |   valid -  0.79104
train -  0.80313   |   valid -  0.70896
Точность на кросс-валидации: 0.71963
Стандартное отклонение: 0.04791


In [44]:
# DecisionTreeClassifier с max_depth=15
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = DecisionTreeClassifier(max_depth=15, random_state=21)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

train -  0.95301   |   valid -  0.86667
train -  0.94064   |   valid -  0.82963
train -  0.95218   |   valid -  0.88889
train -  0.94312   |   valid -  0.88889
train -  0.94971   |   valid -  0.86667
train -  0.93570   |   valid -  0.80741
train -  0.93817   |   valid -  0.85185
train -  0.96290   |   valid -  0.85926
train -  0.93822   |   valid -  0.88806
train -  0.94893   |   valid -  0.82090
Точность на кросс-валидации: 0.85682
Стандартное отклонение: 0.02780


### Сравнение моделей DecisionTreeClassifier

| max_depth | Средняя точность | Ст. отклонение  |    Комментарий      |
|-----------|------------------|-----------------|---------------------|
|     5     |      0.54448     |     0.04014     |    Недообучена      |
|     7     |      0.65284     |     0.04153	 |    Лучше            |
|    10     |      0.71963     |     0.04791     |    Оптимальна       |
|    15     |      0.85682     |     0.02780     |    Переобученае     |

**Вывод:**  
- При низком max_depth модель недообучена.
- При большом max_depth возможное переобучение (train accuracy сильно выше valid).
- Оптимальный баланс при 7-10.

## Случайный лес

### a. Регуляризация по умолчанию

1. Обучите базовую модель с единственными параметрами n_estimators=50, max_depth=14, random_state=21.
2. Используйте стратифицированную K-фолдовую кросс-валидацию с 10 разбиениями для оценки точности модели.
3. Формат вывода результатов кода, в котором вы обучаете и оцениваете базовую модель, должен быть аналогичен тому, что вы получили для логистической регрессии.

n_estimators — количество деревьев в ансамбле случайного леса. Чем больше деревьев, тем устойчивее и точнее модель, но выше время обучения.

max_depth — максимальная глубина каждого дерева. Чем глубже дерево, тем сложнее модель (может переобучаться). Меньшая глубина — меньше переобучение, но возможно недообучение.

In [ ]:
# RandomForestClassifier с n_estimators=50, max_depth=14
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

### b. Оптимизация параметров регуляризации

1. В новых ячейках попробуйте разные значения параметров max_depth и n_estimators.
2. В качестве бонуса поэкспериментируйте с другими параметрами регуляризации, чтобы найти наилучшую комбинацию.

In [ ]:
# RandomForestClassifier с n_estimators=20, max_depth=10
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = RandomForestClassifier(n_estimators=20, max_depth=10, random_state=21)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

In [ ]:
# RandomForestClassifier с n_estimators=100, max_depth=14
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = RandomForestClassifier(n_estimators=100, max_depth=14, random_state=21)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

In [ ]:
# RandomForestClassifier с n_estimators=50, max_depth=7
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = RandomForestClassifier(n_estimators=50, max_depth=7, random_state=21)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

In [ ]:
# RandomForestClassifier с n_estimators=100, max_depth=10
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=21)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

In [ ]:
# RandomForestClassifier с n_estimators=200, max_depth=10
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []
for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=21)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    valid_acc = model.score(X_val, y_val)
    train_scores.append(train_acc)
    valid_scores.append(valid_acc)
    print(f'train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}')
print(f'Точность на кросс-валидации: {np.mean(valid_scores):.5f}')
print(f'Стандартное отклонение: {np.std(valid_scores):.5f}')

| n_estimators | max_depth | Точность на кросс-валидации | Ст. отклонение | Комментарий         |
|--------------|-----------|-----------------------------|----------------|---------------------|
| 20           | 10        | 0.78118                     | 0.03689        | Хорошо              |
| 50           | 7         | 0.68547                     | 0.04434        | Недообучение        |
| 100          | 10        | 0.80713                     | 0.03365        | Лучше               |
| 100          | 14        | 0.89022                     | 0.02668        | Возможное переобуч. |
| 200          | 10        | 0.81231                     | 0.03109        | Хорошо              |

Вывод:

Наилучшая точность — при n_estimators=100, max_depth=14, но возможное переобучение.  
Оптимальные значения: n_estimators=100–200, max_depth=10–14.  
Для финальной модели лучше выбрать параметры с высокой точностью и невысоким переобучением (например, 100/200 и 10).

## 6. Предсказания

1. Выберите лучшую модель (RandomForestClassifier с оптимальными параметрами из наших тестов) и используйте её для предсказаний на тестовом наборе данных (X_test).
2. Посчитайте итоговую точность (accuracy_score).
3. Проанализируйте, для какого дня недели ваша модель делает больше всего ошибок (в процентах от общего числа образцов этого класса в тестовом наборе).
4. Сохраните модель. (joblib.dump)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
# 1. Обучаем лучшую модель на всей обучающей выборке
best_model = RandomForestClassifier(n_estimators=100, max_depth=14, random_state=21)
best_model.fit(X_train, y_train)

In [ ]:
# 2. Предсказания на тесте
y_pred = best_model.predict(X_test)

In [ ]:
# 3. Итоговая точность
test_acc = accuracy_score(y_test, y_pred)
print(f'Точность на тесте: {test_acc:.5f}')

In [ ]:
# 4. Анализ ошибок по классам
cm = confusion_matrix(y_test, y_pred, labels=np.unique(y_test))
errors = (cm.sum(axis=1) - np.diag(cm)) / cm.sum(axis=1)
for i, day in enumerate(np.unique(y_test)):
    print(f'Доля ошибок для {day}: {errors[i]:.2%}')

In [ ]:
# 5. Сохраняем модель
joblib.dump(best_model, '../data/best_random_forest.joblib')

### Сравнение моделей и анализ ошибок

**Сравнение точности:**
- Логистическая регрессия: ~0.60
- SVM (оптимальные параметры): ~0.72–0.75
- DecisionTreeClassifier (оптимальные параметры): ~0.72–0.86
- RandomForestClassifier (лучшие параметры): **0.89645** на тесте

Модель случайного леса с параметрами `n_estimators=100, max_depth=14` показала наивысшую точность среди всех протестированных моделей.

**Анализ ошибок по дням недели:**
- Наибольшая доля ошибок — для класса **0** (25.93%). Это может говорить о сложности распознавания этого дня недели, либо о недостатке признаков для его выделения.
- Следующие по ошибкам: класс **4** (19.05%) и класс **1** (14.55%).
- Минимальные ошибки — для классов **3** (2.50%), **5** (7.41%), **2** (6.67%), **6** (11.27%).  
(0 - 6 дни недели)

**Вывод:**  
Модель хорошо справляется с большинством классов, но для некоторых дней недели (особенно 0 и 4) ошибки выше среднего. Это может быть связано с пересечением признаков между днями или дисбалансом данных. Для дальнейшего улучшения можно попробовать увеличить количество признаков или использовать методы балансировки